# ML Frameworks Side by Side — scikit-learn vs Keras vs PyTorch

**One problem, three frameworks.** We classify Iris flowers (4 features → 3 classes) three
times with an *identical* model — one hidden layer of 16 ReLU units — so that the only thing
that varies is the API.

| Framework | Core abstraction | Training code | You control |
|---|---|---|---|
| **scikit-learn** | Estimator (`fit`/`predict`) | 1 line | almost nothing |
| **Keras** | Layer + Model (`compile`/`fit`) | 2 calls | architecture, optimizer, loss |
| **PyTorch** | `nn.Module` + explicit loop | ~8 lines | everything |

### What to watch for

Three API differences cause most of the bugs when moving between frameworks:

1. **Softmax placement** — Keras puts it in the last layer; PyTorch does **not**
   (`nn.CrossEntropyLoss` applies log-softmax internally for numerical stability).
   Applying it twice quietly degrades training.
2. **Train/eval mode** — PyTorch needs explicit `model.train()` / `model.eval()`;
   Keras handles it inside `fit()`.
3. **Gradient state** — PyTorch *accumulates* gradients, so `optimizer.zero_grad()`
   every step is mandatory.

📖 Theory: [`guide/13_frameworks.md`](../guide/13_frameworks.md) ·
[`guide/05_deep_learning.md`](../guide/05_deep_learning.md)

> **Note:** TensorFlow and PyTorch are optional. Any framework that isn't installed is skipped,
> so the notebook runs end-to-end in a minimal environment.

## Setup

Install what you need (skip any you already have):

```bash
pip install scikit-learn pandas matplotlib      # required
pip install tensorflow                          # for the Keras section
pip install torch                               # for the PyTorch section
```

> ### ⚠️ If the kernel dies partway through
>
> **TensorFlow and PyTorch each bundle their own OpenMP runtime**, and on some
> platforms loading both into one process crashes the kernel (a segfault, usually at
> the first PyTorch computation after TensorFlow has trained). This notebook imports
> torch first to reduce the risk, but on some builds the clash is unavoidable.
>
> **If it happens:** restart the kernel and run *one* deep learning section per
> session — cells 1–2 plus **either** the Keras section **or** the PyTorch section.
> Every section is written to run independently, and any framework that isn't
> importable is skipped cleanly rather than erroring.
>
> This is not a quirk of this notebook — it's a real deployment constraint. It's one
> practical reason production images normally pin **one** deep learning framework
> rather than installing both.

In [ ]:
import os
import time
import warnings

# --- Environment setup: MUST run before importing TensorFlow or PyTorch ---
# TensorFlow and PyTorch each bundle their own OpenMP runtime. Loading both into
# one process can segfault or oversubscribe CPU threads. Capping thread counts up
# front avoids the clash; on this toy dataset it costs no measurable speed.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # silence TF's C++ logging
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")  # deterministic, avoids oneDNN paths
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
warnings.filterwarnings("ignore")

# Import PyTorch FIRST, before TensorFlow/Keras.
# Both ship their own OpenMP runtime. Whichever is imported second loses, and in
# some environments the first PyTorch computation after TensorFlow has trained
# segfaults the kernel. Importing torch first avoids the clash entirely.
try:
    import torch
except ImportError:
    torch = None

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Shared configuration - identical across all three frameworks so the
# comparison is apples to apples.
SEED = 42
HIDDEN = 16          # units in the single hidden layer
EPOCHS = 200
BATCH_SIZE = 16
LEARNING_RATE = 1e-2

np.random.seed(SEED)
results = {}         # framework -> (accuracy, seconds)

print("Available frameworks:")
for name, module in [("scikit-learn", "sklearn"), ("TensorFlow/Keras", "keras"), ("PyTorch", "torch")]:
    try:
        m = __import__(module)
        print(f"  [ok]   {name:<18} {m.__version__}")
    except ImportError:
        print(f"  [skip] {name:<18} not installed")

## Data

Iris: 150 samples, 4 numeric features, 3 balanced classes. Deliberately trivial — the point is
the API, not the modelling challenge.

`stratify=y` keeps class proportions identical across the split. On balanced Iris it barely
matters, but it should be your default reflex — on imbalanced data, an unstratified split can
produce a fold missing a class entirely.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print(f"Features: {iris.feature_names}")
print(f"Classes : {list(iris.target_names)}")

pd.DataFrame(X_train, columns=iris.feature_names).describe().round(2)

### Scaling

Neural networks need scaled inputs — unscaled features make the loss surface badly conditioned
and gradient descent zig-zags ([Module 01 §1.8](../guide/01_math_foundations.md#18-eigenvalues-and-eigenvectors)).

Note **`fit_transform` on train, `transform` on test**. Fitting the scaler on the test set
leaks its statistics into preprocessing — one of the most common forms of data leakage
([Module 02 §5](../guide/02_ml_fundamentals.md#data-leakage--the-1-cause-of-great-model-terrible-production)).

scikit-learn will do this *inside* its Pipeline; Keras and PyTorch need it done up front.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform on TRAIN
X_test_s = scaler.transform(X_test)         # transform ONLY on TEST

print(f"Before scaling - train mean: {X_train.mean(axis=0).round(2)}")
print(f"After  scaling - train mean: {X_train_s.mean(axis=0).round(2)}  (~0)")
print(f"After  scaling - train std : {X_train_s.std(axis=0).round(2)}  (~1)")

---
## 1. scikit-learn

**The Estimator API.** Every object implements the same contract — `fit`, `predict`,
`transform` — which is why the library is so quick to learn.

The important part below is the **`Pipeline`**, not the `MLPClassifier`:

- The scaler lives *inside* the pipeline, so during cross-validation it is re-fit on the
  training portion of each fold → preprocessing leakage becomes structurally impossible.
- The saved artifact contains preprocessing **and** the model, so serving code cannot scale
  features differently than training did — the training/serving skew bug from
  [Module 09](../guide/09_mlops_and_deployment.md) is eliminated by construction.

> `MLPClassifier` is a *toy* neural network: CPU-only, no custom architectures, no
> convolutions. It's here for a like-for-like comparison. For real tabular work you'd use
> `HistGradientBoostingClassifier` or LightGBM — and they'd likely beat all three models here.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sk_model = Pipeline([
    ("scaler", StandardScaler()),                    # preprocessing INSIDE the pipeline
    ("clf", MLPClassifier(
        hidden_layer_sizes=(HIDDEN,),
        activation="relu",
        solver="adam",
        learning_rate_init=LEARNING_RATE,
        max_iter=2000,
        random_state=SEED,
    )),
])

t0 = time.perf_counter()
sk_model.fit(X_train, y_train)          # <-- training is literally one line
sk_time = time.perf_counter() - t0

y_pred_sk = sk_model.predict(X_test)
sk_acc = accuracy_score(y_test, y_pred_sk)
results["scikit-learn"] = (sk_acc, sk_time)

print(f"Test accuracy : {sk_acc:.4f}")
print(f"Training time : {sk_time:.2f}s")
print(f"Iterations    : {sk_model.named_steps['clf'].n_iter_}")

`predict_proba` gives calibrated-ish class probabilities — useful when a downstream decision
needs a threshold rather than a hard label.

In [ ]:
print("Classification report:")
print(classification_report(y_test, y_pred_sk, target_names=iris.target_names, digits=3))

print("Confusion matrix (rows = actual, cols = predicted):")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred_sk),
    index=[f"actual {c}" for c in iris.target_names],
    columns=[f"pred {c}" for c in iris.target_names],
))

print("\nFirst 5 predicted probabilities:")
print(pd.DataFrame(
    sk_model.predict_proba(X_test)[:5].round(3), columns=iris.target_names
))

---
## 2. Keras

**Sequential model + `compile` → `fit`.** Callbacks give you early stopping, LR scheduling and
checkpointing for free — in PyTorch you write all of that yourself.

### The loss/activation pairing (the #1 Keras bug)

| Task | Final layer | Loss | Labels |
|---|---|---|---|
| Binary | `Dense(1, "sigmoid")` | `binary_crossentropy` | 0/1 |
| **Multiclass** | **`Dense(K, "softmax")`** | **`sparse_categorical_crossentropy`** | **integers** |
| Multiclass (one-hot) | `Dense(K, "softmax")` | `categorical_crossentropy` | one-hot |
| Regression | `Dense(1)` (no activation) | `mse` / `huber` | float |

Using `categorical_crossentropy` with *integer* labels doesn't raise a clear error — it
silently trains on garbage. We have integer labels, so we use the **`sparse_`** variant.

In [ ]:
keras_history = None
try:
    import keras
    from keras import layers

    keras.utils.set_random_seed(SEED)

    keras_model = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(HIDDEN, activation="relu"),
        layers.Dense(3, activation="softmax"),        # <-- softmax IS in the model
    ], name="iris_keras")

    keras_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",       # "sparse" = integer labels
        metrics=["accuracy"],
    )
    keras_model.summary()
except ImportError:
    keras_model = None
    print("Keras not installed - skipping. Install with: pip install tensorflow")

In [ ]:
if keras_model is not None:
    t0 = time.perf_counter()
    keras_history = keras_model.fit(
        X_train_s, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=20, restore_best_weights=True
        )],
        verbose=0,
    )
    keras_time = time.perf_counter() - t0

    keras_loss, keras_acc = keras_model.evaluate(X_test_s, y_test, verbose=0)
    results["Keras"] = (keras_acc, keras_time)

    print(f"Test accuracy : {keras_acc:.4f}")
    print(f"Training time : {keras_time:.2f}s")
    print(f"Epochs run    : {len(keras_history.history['loss'])} (early stopping from {EPOCHS})")
    print(f"Parameters    : {keras_model.count_params():,}")

### Training curves

Keras returns a `History` object — the cheapest diagnostic you have. Read it for the patterns
from [Module 02 §3](../guide/02_ml_fundamentals.md#3-the-biasvariance-tradeoff):

- **Both curves high and flat** → underfitting (high bias) → bigger model, train longer.
- **Train ≪ validation** → overfitting (high variance) → more data or regularization.
- **Validation turning upward** → stop training. That's exactly what `EarlyStopping` detected.

In [ ]:
if keras_history is not None:
    h = keras_history.history
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(h["loss"], label="train")
    axes[0].plot(h["val_loss"], label="validation")
    axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(h["accuracy"], label="train")
    axes[1].plot(h["val_accuracy"], label="validation")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle("Keras training history")
    plt.tight_layout()
    plt.show()
else:
    print("Keras not available - no history to plot.")

---
## 3. PyTorch

**`nn.Module` + a training loop you write yourself.** Maximum control, maximum boilerplate.

The six-step inner loop is the pattern behind *every* PyTorch script you will ever read —
including LLM training code:

```
1. optimizer.zero_grad()   clear stale gradients   <- forget this and the model won't learn
2. logits = model(xb)      forward
3. loss = criterion(...)   loss
4. loss.backward()         backward (populates .grad)
5. clip_grad_norm_(...)    clip (optional but cheap insurance)
6. optimizer.step()        update
```

Two things to notice in the model definition:

- **No softmax.** `nn.CrossEntropyLoss` applies log-softmax internally
  ([Module 01 §7.2](../guide/01_math_foundations.md#72-numerical-stability-tricks)).
  Adding your own applies it twice.
- **Labels must be `torch.long`** (int64), or you get a cryptic dtype error.

In [ ]:
torch_model = None
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader

    torch.manual_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    Xtr = torch.tensor(X_train_s, dtype=torch.float32)
    ytr = torch.tensor(y_train, dtype=torch.long)         # <-- long, not int32/float
    Xte = torch.tensor(X_test_s, dtype=torch.float32).to(device)
    yte = torch.tensor(y_test, dtype=torch.long).to(device)

    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=BATCH_SIZE, shuffle=True)

    torch_model = nn.Sequential(
        nn.Linear(Xtr.shape[1], HIDDEN),
        nn.ReLU(),
        nn.Linear(HIDDEN, 3),        # <-- NO softmax: CrossEntropyLoss wants raw logits
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(torch_model.parameters(), lr=LEARNING_RATE)

    print(torch_model)
    print(f"\nDevice     : {device}")
    print(f"Parameters : {sum(p.numel() for p in torch_model.parameters()):,}")
except ImportError:
    print("PyTorch not installed - skipping. Install with: pip install torch")

In [ ]:
if torch_model is not None:
    torch_losses = []

    t0 = time.perf_counter()
    for epoch in range(EPOCHS):
        torch_model.train()                                  # enable dropout / BN batch stats
        epoch_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()                            # 1. clear stale gradients
            logits = torch_model(xb)                         # 2. forward
            loss = criterion(logits, yb)                     # 3. loss
            loss.backward()                                  # 4. backward
            nn.utils.clip_grad_norm_(torch_model.parameters(), 1.0)   # 5. clip
            optimizer.step()                                 # 6. update

            epoch_loss += loss.item() * len(yb)              # .item() -> don't retain the graph
        torch_losses.append(epoch_loss / len(loader.dataset))

    torch_time = time.perf_counter() - t0

    torch_model.eval()                                       # disable dropout / BN running stats
    with torch.no_grad():                                    # no autograd graph -> less memory
        torch_acc = (torch_model(Xte).argmax(dim=1) == yte).float().mean().item()

    results["PyTorch"] = (torch_acc, torch_time)
    print(f"Test accuracy : {torch_acc:.4f}")
    print(f"Training time : {torch_time:.2f}s")
    print(f"Final loss    : {torch_losses[-1]:.4f}")

In [ ]:
if torch_model is not None:
    plt.figure(figsize=(6, 4))
    plt.plot(torch_losses)
    plt.title("PyTorch training loss (you log this yourself)")
    plt.xlabel("epoch"); plt.ylabel("cross-entropy loss")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

### The three lines everyone forgets

| Missing | Symptom |
|---|---|
| `optimizer.zero_grad()` | Gradients accumulate across batches; the model barely learns |
| `model.train()` / `model.eval()` | Dropout and BatchNorm behave wrongly at inference |
| `torch.no_grad()` at eval | Memory blows up; slower; possible OOM |

Our model has no dropout or batch norm, so `train()`/`eval()` changes nothing *here* — but the
habit is what saves you later. Forgetting `model.eval()` is the single most common PyTorch bug
in production.

---
## Comparison

In [ ]:
summary = pd.DataFrame([
    {
        "Framework": name,
        "Test accuracy": f"{acc:.4f}",
        "Train time (s)": round(t, 2),
        "Training code": {
            "scikit-learn": "1 line (.fit)",
            "Keras": "2 calls (compile + fit)",
            "PyTorch": "explicit loop (~8 lines)",
        }[name],
        "Early stopping": {
            "scikit-learn": "n_iter_no_change",
            "Keras": "built-in callback",
            "PyTorch": "you write it",
        }[name],
    }
    for name, (acc, t) in results.items()
])
summary

In [ ]:
if len(results) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = list(results)
    accs = [results[n][0] for n in names]
    times = [results[n][1] for n in names]

    axes[0].bar(names, accs)
    axes[0].set_ylim(0, 1.05); axes[0].set_title("Test accuracy")
    for i, v in enumerate(accs):
        axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center")

    axes[1].bar(names, times, color="darkorange")
    axes[1].set_title("Training time (s)")
    for i, v in enumerate(times):
        axes[1].text(i, v, f"{v:.2f}s", ha="center", va="bottom")

    plt.suptitle("Same model, same data - the frameworks differ in API, not capability")
    plt.tight_layout()
    plt.show()

## Takeaway

Same architecture, same data, near-identical accuracy. The frameworks differ in **how much you
must write and how much you can control** — not in capability at this scale.

**Pick the lowest-control tool that solves your problem.**

| Your problem | Reach for |
|---|---|
| Tabular data, < ~10M rows | **scikit-learn / LightGBM** — not a neural network |
| Credit scoring, regulated decisions | **scikit-learn** logistic regression (interpretable) |
| Fast DL prototype, mobile deployment | **Keras** → TFLite |
| Research, custom architectures, LLMs | **PyTorch** |
| Anything with a pretrained model | **Hugging Face** (PyTorch underneath) |

Reaching for PyTorch to fit a linear model on 500 rows is an error of judgment, not a display
of skill.

### API Rosetta stone

| Concept | scikit-learn | Keras | PyTorch |
|---|---|---|---|
| Dense layer | — | `layers.Dense(64)` | `nn.Linear(in, 64)` |
| Train | `.fit(X, y)` | `.fit(X, y, epochs=n)` | explicit loop |
| Predict | `.predict(X)` | `.predict(X)` | `model(x)` in `no_grad()` |
| Probabilities | `.predict_proba(X)` | `.predict(X)` (if softmax) | `softmax(logits, dim=1)` |
| Multiclass loss | internal | `"sparse_categorical_crossentropy"` | `nn.CrossEntropyLoss()` |
| Train/eval mode | N/A | automatic in `fit` | `.train()` / `.eval()` |
| Save | `joblib.dump(m, f)` | `m.save("m.keras")` | `torch.save(m.state_dict(), f)` |
| GPU | ❌ CPU only | automatic | `.to("cuda")` explicitly |

---

## Exercises

1. **Break the softmax rule.** Add `nn.Softmax(dim=1)` as a final PyTorch layer, keeping
   `CrossEntropyLoss`. Retrain. Does accuracy drop? Why does double log-softmax hurt?
2. **Break the Keras loss pairing.** Switch to `categorical_crossentropy` while keeping integer
   labels. What happens — an error, or silently wrong training?
3. **Remove `optimizer.zero_grad()`** from the PyTorch loop and plot the loss. Explain the shape.
4. **Beat all three.** Fit a `HistGradientBoostingClassifier` on the raw (unscaled) features.
   How does accuracy compare, and how long did it take you to write?
5. **Add proper validation.** Replace the single train/test split with `StratifiedKFold` on the
   scikit-learn pipeline and report mean ± std. Why does the spread matter more than the mean?

📖 Continue: [`guide/13_frameworks.md`](../guide/13_frameworks.md) ·
[`guide/09_mlops_and_deployment.md`](../guide/09_mlops_and_deployment.md)